In [1]:
import pandas as pd
import os 

os.chdir('/mnt/sdg/robert/deepwidegp/ARD-DeepWideGP')

In [4]:
ukb_snps = pd.read_csv("data/1-inputs/SNP_array_overlap/UKB_SNP_array.csv")


In [2]:
from pathlib import Path

# Path to the file
finngen_sumstats_path = Path("data/1-inputs/SNP_array_overlap/summary_stats_release_finngen_R12_AB1TUBERCU_MILIARY.gz")

# Read the gzipped summary stats
finngen_sumstats = pd.read_csv(
    finngen_sumstats_path,
    compression="gzip",   # or 'infer', but this is explicit
    sep="\t",             # FinnGen sumstats are tab-separated
    header=0              # first line is header
)

# Quick sanity checks
print(f"Shape: {finngen_sumstats.shape}")
print("Columns:", finngen_sumstats.columns.tolist())
print(finngen_sumstats.head())

/tmp/ipykernel_835573/3983975465.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  finngen_sumstats = pd.read_csv(


Shape: (21326999, 13)
Columns: ['#chrom', 'pos', 'ref', 'alt', 'rsids', 'nearest_genes', 'pval', 'mlogp', 'beta', 'sebeta', 'af_alt', 'af_alt_cases', 'af_alt_controls']
   #chrom    pos ref alt         rsids nearest_genes      pval     mlogp  \
0       1  13668   G   A     rs2691328       DDX11L1  0.977107  0.010058   
1       1  14506   G   A  rs1240557819        WASH7P  0.810669  0.091156   
2       1  14521   C   T  rs1378626194        WASH7P  0.627701  0.202247   
3       1  14717   G   A   rs377122907        WASH7P  0.779895  0.107964   
4       1  14773   C   T   rs878915777        WASH7P  0.315651  0.500793   

       beta    sebeta    af_alt  af_alt_cases  af_alt_controls  
0 -0.065434   2.28022  0.005985      0.005686         0.005985  
1 -0.560071   2.33789  0.004639      0.003856         0.004639  
2 -2.638710   5.44104  0.001084      0.000098         0.001084  
3 -3.416550  12.22570  0.000237      0.000000         0.000237  
4  1.200240   1.19613  0.014214      0.022092    

In [5]:
ukb_snps.head()

,chrom,snp,cm,pos,a1,a2
0,9,rs202245356,0.0,16967,A,G
1,9,rs9407029,0.0,45440,C,A
2,9,rs73638076,0.0,113128,T,G
3,9,rs4742632,0.0,116832,T,C
4,9,rs4742643,0.0,117428,G,C


In [ ]:
finngen_sumstats.head()

In [ ]:
ukb_mrg = ukb_snps.copy()
ukb_mrg.columns
ukb_mrg['chrom:pos'] = ukb_mrg['chrom'].astype(str) + ':' + ukb_mrg['pos'].astype(str)
ukb_mrg.head()

In [14]:
#dang we must convert to GRCH37 ffrom GRCH38 first
df = finngen_sumstats.copy()

# Normalise column names
df.columns = [c.lower() for c in df.columns]

# Rename to standard names if needed
rename_map = {}
for col in ["#chrom", "chromosome"]:
    if col in df.columns:
        rename_map[col] = "chr"
for col in ["position", "bp"]:
    if col in df.columns:
        rename_map[col] = "pos"
df = df.rename(columns=rename_map)

required = ["chr", "pos", "ref", "alt"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in sumstats: {missing}")

# Build a minimal VCF-like DataFrame
vcf_df = pd.DataFrame({
    "CHROM": df["chr"],
    "POS": df["pos"],
    "ID": df.get("rsid", df.get("rsids", ".")),
    "REF": df["ref"],
    "ALT": df["alt"],
    "QUAL": ".",
    "FILTER": ".",
    "INFO": "."
})


In [15]:
vcf_path = Path("data/1-inputs/SNP_array_overlap/AB1TUBERCU_MILIARY_GRCh38.vcf")

with vcf_path.open("w") as f:
    # Minimal header
    f.write("##fileformat=VCFv4.2\n")
    f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
    vcf_df.to_csv(f, sep="\t", index=False, header=False)


In [11]:
overlap = pd.merge(ukb_mrg, finngen_mrg, on='chrom:pos', how='inner')
print(len(overlap), "overlapping variants between UKB and finngen array")
print(finngen_mrg['chrom:pos'].nunique(), "unqie variants in finngen array")
print(ukb_mrg['chrom:pos'].nunique(), "unique variants in UKB BIM files")

13058 overlapping variants between UKB and finngen array
20915226 unqie variants in finngen array
805426 unique variants in UKB BIM files
